In [1]:
import os
import glob
import requests
import zipfile
from tqdm.auto import tqdm

In [2]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "HAM10000_Skin_Lesion_Classification"
)

folders = [
    "data/raw",
    "data/processed",
    "data/splits",
    "notebooks",
    "models",
    "src",
    "reports/figures",
    "reports/tables",
    "app"
]

for folder in folders:
  os.makedirs(
      os.path.join(PROJECT_PATH, folder),
      exist_ok=True
  )

print("Project folders created successfully!")

Project folders created successfully!


In [3]:
print(os.listdir(PROJECT_PATH))

['data', 'notebooks', 'models', 'reports', 'app', 'src']


In [4]:
RAW_PATH = os.path.join(PROJECT_PATH, 'data', 'raw')
os.makedirs(RAW_PATH, exist_ok=True)

BASE_URL = "https://dataverse.harvard.edu"
PERSISTENT_ID = "doi:10.7910/DVN/DBW86T"

api_url = (
    f"{BASE_URL}/api/datasets/:persistentId/versions/:latest"
    f"?persistentId={PERSISTENT_ID}"
)

# Adding a User-Agent header to prevent 403 Forbidden responses
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(api_url, headers=headers)
response.raise_for_status()
dataset_info = response.json()

print("Dataset information retrieved!")

Dataset information retrieved!


In [5]:
files = dataset_info['data']['files']

for f in files:
  print(f['dataFile']['filename'] )
  print('ID:', f['dataFile']['id'])
  print("-" * 40)

HAM10000_images_part_1.zip
ID: 3172585
----------------------------------------
HAM10000_images_part_2.zip
ID: 3172584
----------------------------------------
HAM10000_metadata.tab
ID: 4338392
----------------------------------------
HAM10000_segmentations_lesion_tschandl.zip
ID: 3838943
----------------------------------------
ISIC2018_Task3_Test_GroundTruth.tab
ID: 6924466
----------------------------------------
ISIC2018_Task3_Test_Images.zip
ID: 3855824
----------------------------------------
ISIC2018_Task3_Test_NatureMedicine_AI_Interaction_Benefit.tab
ID: 3864681
----------------------------------------


In [6]:
target_file = [
    "HAM10000_images_part_1.zip",
    "HAM10000_images_part_2.zip",
    "HAM10000_metadata.tab"
]

for f in files:
  filename = f['dataFile']['filename']

  if filename not in target_file:
    continue

  file_id = f['dataFile']['id']
  download_url = (
        f"{BASE_URL}/api/access/datafile/{file_id}"
    )

  save_path = os.path.join(RAW_PATH, filename)

  with requests.get(download_url,stream=True, timeout=(30, 600)) as r:
    r.raise_for_status()

    total_size = int(r.headers.get('content-length', 0))
    with open(save_path, 'wb') as f:
      with tqdm(
          total=total_size,
          unit='B',
          unit_scale=True,
          desc=filename,
          leave=True ) as progress:

          for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
              f.write(chunk)
              progress.update(len(chunk))

print(f"Downloaded: {filename}")





HAM10000_images_part_1.zip:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

HAM10000_images_part_2.zip:   0%|          | 0.00/1.40G [00:00<?, ?B/s]

HAM10000_metadata.tab:   0%|          | 0.00/830k [00:00<?, ?B/s]

Downloaded: ISIC2018_Task3_Test_NatureMedicine_AI_Interaction_Benefit.tab


In [7]:
for filename in target_file[:2]:
  zip_path = os.path.join(RAW_PATH, filename)

  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(RAW_PATH)

print("Extracted files successfully!")

Extracted files successfully!


In [8]:
for root, dirs, files in os.walk(RAW_PATH):
  level = root.replace(RAW_PATH, '').count(os.sep)

  if level > 2:
    dirs[:] = []
    continue
  indent = ' ' * 4 * level
  print(f"{indent}{os.path.basename(root)}/")
  subindent = ' ' * 4 * (level + 1)

  for f in files[:5]:
    print(f"{subindent}{f}")

raw/
    ISIC_0033321.jpg
    ISIC_0033322.jpg
    ISIC_0033323.jpg
    ISIC_0033324.jpg
    ISIC_0033325.jpg


In [9]:
image_files = glob.glob(
    os.path.join(RAW_PATH, '**', '*.jpg'),
    recursive=True
)
print("Total images:", len(image_files))
print("Expected images: 10015")

assert len(image_files) == 10015, (
    "Image count does not match expected dataset size!"
)

Total images: 10015
Expected images: 10015
